In [47]:
from pathlib import Path
import re
import fitz
import pymupdf

pdf_folder = Path("expenses")
pdf_files = list(pdf_folder.glob("*.pdf"))

def extract_pages(pdf_path):
    doc = pymupdf.open(pdf_path)

    pages = []

    for page in doc:
        pages.append(page.get_text())

    return pages

In [48]:
pages = extract_pages(pdf_files[0])

print("Number of pages:", len(pages))
# print(text[:2000])

Number of pages: 5


In [49]:
pages

['  \nHamburger Sparkasse\nKontoauszug 2/2026\nHaspaJoker 1507465423, DE36 2005 0550 1507 4654 23 \n1. April 2026\nSeite 1 von 5\nFrau\nAni Khvadagiani\nHeidberg 4\n22301 Hamburg\nFiliale\nWinterhude\nMühlenkamp 34\n22303 Hamburg\nTelefon 040 3578-95570\nFax 040 3578-94477\nmuehlenkamp@haspa.de\nDatum\nErläuterung\nBetrag EUR\nKontostand am 27.02.2026, Auszug Nr.    1\n               -2,30\n16.03.2026 Zahlungseingang\nAni Khvadagiani\n                1,00\n18.03.2026 Zahlungseingang\nAni Khvadagiani Konto 1004765143 Ausgleich des Kont osaldos wegen \nKontoauflösung\n            1.365,42\n18.03.2026 Bargeldauszahlung\n             -100,00\n23.03.2026 Überw. OnlineBanking\nHAMBURGER HOCHBAHN AG-20095 FP-Nr.8004992556 DATUM 21.03.2026, 13.42 \nUHR 281716909-20260321134144\n              -60,00\n23.03.2026 Überw. OnlineBanking\nAvani Lakhe DATUM 21.03.2026, 13.45 UHR 1034693648-20260321134546\n              -50,00\n23.03.2026 Kartenzahlung\nREWE SAGT DANKE. 41400035/Moorfuhrt w/Hamburg /DE

In [50]:
for i, page in enumerate(pages):
    print(f"\n===== PAGE {i + 1} =====")
    print("START:")
    print(page[:500])

    print("\nEND:")
    print(page[-500:])


===== PAGE 1 =====
START:
  
Hamburger Sparkasse
Kontoauszug 2/2026
HaspaJoker 1507465423, DE36 2005 0550 1507 4654 23 
1. April 2026
Seite 1 von 5
Frau
Ani Khvadagiani
Heidberg 4
22301 Hamburg
Filiale
Winterhude
Mühlenkamp 34
22303 Hamburg
Telefon 040 3578-95570
Fax 040 3578-94477
muehlenkamp@haspa.de
Datum
Erläuterung
Betrag EUR
Kontostand am 27.02.2026, Auszug Nr.    1
               -2,30
16.03.2026 Zahlungseingang
Ani Khvadagiani
                1,00
18.03.2026 Zahlungseingang
Ani Khvadagiani Konto 1004765143 Ausgle

END:
AGT DANKE. 41400035/Moorfuhrt w/Hamburg /DE 2026-03-23T12:46 
Debitk.10 2030-12
              -32,23
Übertrag:
            1.047,52
Hamburger Sparkasse AG
Dammtorstraße 1
20354 Hamburg
www.haspa.de
Vorsitzender des Aufsichtsrats:
Prof. Dr. Burkhard Schwenker
Vorstand: Dr. Harald Vogelsang, 
Dr. Olaf Oesterhelweg, Axel Kodlin,
Jürgen Marquardt, Birte Quitt
Sitz Hamburg
AG Hamburg HRB 80 691
USt-ID-Nr. DE216540952
BLZ 200 505 50
BIC HASPDEHHXXX
Tel. 040 3578-0
Fax

In [51]:
def is_transaction_page(page_text):
    return (
        "Datum" in page_text
        and "Erläuterung" in page_text
        and "Betrag EUR" in page_text
    )

In [52]:
transaction_pages = []

for page in pages:
    if is_transaction_page(page):
        transaction_pages.append(page)

print(len(transaction_pages))

3


In [53]:
for i, page in enumerate(pages):
    print(i + 1, is_transaction_page(page))

1 True
2 True
3 True
4 False
5 False


In [54]:
def remove_footer(page_text):
    footer_marker = "Hamburger Sparkasse AG"

    if footer_marker in page_text:
        page_text = page_text.split(footer_marker)[0]

    return page_text

In [55]:
cleaned_pages = []

for page in transaction_pages:
    cleaned = remove_footer(page)
    cleaned_pages.append(cleaned)

In [56]:
for i, page in enumerate(cleaned_pages):
    print(f"\n--- PAGE {i + 1} END ---")
    print(page[-500:])


--- PAGE 1 END ---
 -9,40
24.03.2026 Kartenzahlung
Revolut..4446./70 SIR JOHN ROGERSON .S QUAY/Dublin/IE/1 2026-03-22T21:48 
Debitk.1 2030-12 Zahl.System DebitMastercard
              -15,00
24.03.2026 Kartenzahlung
Revolut..4446./70 SIR JOHN ROGERSON .S QUAY/Dublin/IE/1 2026-03-23T12:26 
Debitk.1 2030-12 Zahl.System DebitMastercard
              -16,00
24.03.2026 Kartenzahlung
REWE SAGT DANKE. 41400035/Moorfuhrt w/Hamburg /DE 2026-03-23T12:46 
Debitk.10 2030-12
              -32,23
Übertrag:
            1.047,52


--- PAGE 2 END ---
ss-Ring 9/089964600/DE/0 2026-03-28T00:00 
Debitk.10 2030-12 Teillieferung(Final) Zahl.System DebitMastercard
              -54,00
30.03.2026 Kartenzahlung
Pallas.WorldGmbH+Co.KG/Neuer Pferde markt 13/Hamburg/DE/0 2026-03-26T20:51 
Debitk.10 2030-12 Zahl.System DebitMastercard
               -4,18
30.03.2026 Kartenzahlung
SumUp .Cafe Eppendorf/Eppendorfer M artplatz 2/Hamburg/DE/0 2026-03-27T20:26 
Debitk.10 2030-12 Zahl.System DebitMastercard
           

In [57]:
def split_transactions(page_text):
    pieces = re.split(
        r"(?=^\d{2}\.\d{2}\.\d{4} .+$)",
        page_text,
        flags=re.MULTILINE
    )

    return pieces

In [58]:
pieces = split_transactions(cleaned_pages[0])

print("Pieces:", len(pieces))

for i, piece in enumerate(pieces):
    print(f"\n--- PIECE {i} ---")
    print(piece[:300])

Pieces: 13

--- PIECE 0 ---
  
Hamburger Sparkasse
Kontoauszug 2/2026
HaspaJoker 1507465423, DE36 2005 0550 1507 4654 23 
1. April 2026
Seite 1 von 5
Frau
Ani Khvadagiani
Heidberg 4
22301 Hamburg
Filiale
Winterhude
Mühlenkamp 34
22303 Hamburg
Telefon 040 3578-95570
Fax 040 3578-94477
muehlenkamp@haspa.de
Datum
Erläuterung
Betr

--- PIECE 1 ---
16.03.2026 Zahlungseingang
Ani Khvadagiani
                1,00


--- PIECE 2 ---
18.03.2026 Zahlungseingang
Ani Khvadagiani Konto 1004765143 Ausgleich des Kont osaldos wegen 
Kontoauflösung
            1.365,42


--- PIECE 3 ---
18.03.2026 Bargeldauszahlung
             -100,00


--- PIECE 4 ---
23.03.2026 Überw. OnlineBanking
HAMBURGER HOCHBAHN AG-20095 FP-Nr.8004992556 DATUM 21.03.2026, 13.42 
UHR 281716909-20260321134144
              -60,00


--- PIECE 5 ---
23.03.2026 Überw. OnlineBanking
Avani Lakhe DATUM 21.03.2026, 13.45 UHR 1034693648-20260321134546
              -50,00


--- PIECE 6 ---
23.03.2026 Kartenzahlung
REWE SAGT DANKE. 4140003

In [59]:
def split_transactions(page_text):
    pieces = re.split(
        r"(?=^\d{2}\.\d{2}\.\d{4} .+$)",
        page_text,
        flags=re.MULTILINE
    )

    transactions = []

    for piece in pieces:
        if re.match(r"^\d{2}\.\d{2}\.\d{4}", piece):
            transactions.append(piece)

    return transactions

In [60]:
def parse_transaction(raw):
    header = raw.splitlines()[0]
    date, transaction_type = header.split(maxsplit=1)
    transaction_type = re.split(r"\s*/\s*Wert:", transaction_type, maxsplit=1)[0].strip()

    return {
        "date": date,
        "type": transaction_type,
        "raw": raw,
    }


all_transactions = []

for page in cleaned_pages:
    transactions = split_transactions(page)
    all_transactions.extend(parse_transaction(raw) for raw in transactions)

print("Total transactions:", len(all_transactions))

Total transactions: 41


In [61]:
all_transactions

[{'date': '16.03.2026',
  'type': 'Zahlungseingang',
  'raw': '16.03.2026 Zahlungseingang\nAni Khvadagiani\n                1,00\n'},
 {'date': '18.03.2026',
  'type': 'Zahlungseingang',
  'raw': '18.03.2026 Zahlungseingang\nAni Khvadagiani Konto 1004765143 Ausgleich des Kont osaldos wegen \nKontoauflösung\n            1.365,42\n'},
 {'date': '18.03.2026',
  'type': 'Bargeldauszahlung',
  'raw': '18.03.2026 Bargeldauszahlung\n             -100,00\n'},
 {'date': '23.03.2026',
  'type': 'Überw. OnlineBanking',
  'raw': '23.03.2026 Überw. OnlineBanking\nHAMBURGER HOCHBAHN AG-20095 FP-Nr.8004992556 DATUM 21.03.2026, 13.42 \nUHR 281716909-20260321134144\n              -60,00\n'},
 {'date': '23.03.2026',
  'type': 'Überw. OnlineBanking',
  'raw': '23.03.2026 Überw. OnlineBanking\nAvani Lakhe DATUM 21.03.2026, 13.45 UHR 1034693648-20260321134546\n              -50,00\n'},
 {'date': '23.03.2026',
  'type': 'Kartenzahlung',
  'raw': '23.03.2026 Kartenzahlung\nREWE SAGT DANKE. 41400035/Moorfuhrt

In [63]:
all_transactions[10]


{'date': '24.03.2026',
 'type': 'Kartenzahlung',
 'raw': '24.03.2026 Kartenzahlung\nRevolut..4446./70 SIR JOHN ROGERSON .S QUAY/Dublin/IE/1 2026-03-23T12:26 \nDebitk.1 2030-12 Zahl.System DebitMastercard\n              -16,00\n'}